# 🧲 Kernel-Gradient Drifting 

[![arXiv](https://img.shields.io/badge/arXiv-2605.10727-b31b1b.svg)](https://arxiv.org/abs/2605.10727)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carrask0/kernel-grad-drift/blob/main/demo.ipynb)

A minimal, self-contained introduction: 

1. — Core algorithm
2. — Target distribution and drift field
3. — Train and compare

---

## Setup

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q tqdm
else:
    !pip install -q matplotlib torch tqdm

In [ ]:
import os
import sys
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from functools import partial
from tqdm.auto import tqdm

IN_COLAB = ("google.colab" in sys.modules) or (os.environ.get("COLAB_GPU") is not None)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif (not IN_COLAB) and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

# Colors for plotting
COLOR_BASE = "tab:green"    # vanilla / base drift
COLOR_GRAD = "tab:blue"     # kernel-gradient drift (ours)
COLOR_REF  = "tab:orange"   # reference / data / target samples

# Part 1 — Core algorithm

### The training objective is unchanged

Drifting models (Deng et al. 2026) are **one-step** generators: a network $f_\theta$ maps noise
$\epsilon$ to a sample, and training moves each generated sample toward a
*drifted* version of itself,

$$
\mathcal{L}(\theta)=\mathbb{E}_{\epsilon}\Big[\big\|\,f_\theta(\epsilon)-\mathrm{sg}\big(f_\theta(\epsilon)+\mathbf{V}(f_\theta(\epsilon))\big)\big\|^2\Big].
$$
 **Only the drift field $\mathbf{V}$ changes** in our
work — the loss is identical to vanilla drifting.

In [ ]:
def drifting_loss(gen: torch.Tensor, pos: torch.Tensor, drift_fn):
    with torch.no_grad():
        V = drift_fn(gen, pos, gen) 
        target = (gen + V).detach()
    return ((gen - target) ** 2).mean()

## Base drift vs. kernel-gradient drift

**Vanilla drift** combines a kernel $k(x,y)$ (the *weights*) with the
Euclidean displacement $y-x$ (the *direction*):

$$
\mathbf{V}_{p,q}(x)=\underbrace{\frac{\mathbb{E}_{y\sim p}[k(x,y)\,(y-x)]}{\mathbb{E}_{y\sim p}[k(x,y)]}}_{\text{attraction to data }p}
-\underbrace{\frac{\mathbb{E}_{y'\sim q_\theta}[k(x,y')\,(y'-x)]}{\mathbb{E}_{y'\sim q_\theta}[k(x,y')]}}_{\text{self-repulsion from }q_\theta}.
$$

**Our kernel-gradient drift** unifies both terms, using the gradient of the kernel to obtain weight and direction:

$$
\mathbf{V}^{\nabla}_{p,q}(x)=\frac{\mathbb{E}_{y\sim p}[\nabla_x k(x,y)]}{\mathbb{E}_{y\sim p}[k(x,y)]}
-\frac{\mathbb{E}_{y'\sim q_\theta}[\nabla_x k(x,y')]}{\mathbb{E}_{y'\sim q}[k(x,y')]}
\;=\;\nabla_x\log\frac{\widehat p_k(x)}{\widehat q_k(x)}.
$$

The right-hand identity (paper Prop. 3.1) is an important point: the gradient drift is
exactly the **score difference of the kernel-smoothed densities**, which is
*identifiable* for characteristic kernels (zero drift $\Rightarrow p=q$).

### **Gaussian vs Laplace kernels:**

For a radial kernel $k(x,y)=\kappa(d)$, $d=\|x-y\|$, we have
$\nabla_x k=\kappa'(d)\frac{x-y}{d}$. For the following two cases:

* **Gaussian** $k=e^{-d^2/2\tau^2}$: $\;\nabla_x k=k\,\dfrac{y-x}{\tau^2}$ — *parallel*
  to $y-x$. Base and gradient drift coincide up to a scalar ⟹ **no difference**.
* **Laplace** $k=e^{-d/\tau}$: $\;\nabla_x k=k\,\dfrac{y-x}{\tau\,d}$ — the extra
  $1/d$ down-weights *far* neighbours. The base drift instead mixes raw $y-x$, so
  distant points dominate the **direction** and bias it (paper Fig. 2).

So the gradient correction only bites for **non-Gaussian** kernels (paper Fig. 3). We use the **Laplace kernel** throughout.

In [ ]:
def compute_V(X, X_pos, X_neg, *, temp=0.5, mode="gradient",
              ignore_self_neg=True, max_step=None, min_dist=1e-2, eps=1e-8):
    '''Laplace kernel  k(x, y) = exp(-d / temp),  d = ||x - y||.
    Drift  V = (pull toward X_pos) - (push from X_neg):

      base      :  V = E[ k(x,y) (y - x) ] / E[ k(x,y) ]
      gradient  :  V = E[ grad_x k(x,y) ]  / E[ k(x,y) ],
                   with  grad_x k = k(x,y) (y - x) / (temp d)
    '''
    dist_pos = torch.cdist(X, X_pos)
    dist_neg = torch.cdist(X, X_neg)

    k_pos = torch.exp(-dist_pos / temp)
    k_neg = torch.exp(-dist_neg / temp)

    if ignore_self_neg and X.shape[0] == X_neg.shape[0]:   # don't repel a point from itself
        eye = torch.eye(X.shape[0], device=X.device, dtype=torch.bool)
        k_neg = k_neg.masked_fill(eye, 0.0)

    diff_pos = X_pos.unsqueeze(0) - X.unsqueeze(1)         # (y - x), [Nx, Npos, D]
    diff_neg = X_neg.unsqueeze(0) - X.unsqueeze(1)         # (y - x), [Nx, Nneg, D]

    if mode == "base":                                     
        # V = E[k (y - x)] / E[k]
        V_pos = (k_pos.unsqueeze(-1) * diff_pos).sum(1) / k_pos.sum(1, keepdim=True).clamp_min(eps)
        V_neg = (k_neg.unsqueeze(-1) * diff_neg).sum(1) / k_neg.sum(1, keepdim=True).clamp_min(eps)
    else:                                                  
        # V = E[grad_x k] / E[k] -> grad_x k = k (y - x) / (temp d)
        V_pos = (k_pos.unsqueeze(-1) * diff_pos
                 / (temp * dist_pos.clamp_min(min_dist)).unsqueeze(-1)).sum(1) / k_pos.sum(1, keepdim=True).clamp_min(eps)
        V_neg = (k_neg.unsqueeze(-1) * diff_neg
                 / (temp * dist_neg.clamp_min(min_dist)).unsqueeze(-1)).sum(1) / k_neg.sum(1, keepdim=True).clamp_min(eps)

    V = V_pos - V_neg

    if max_step is not None:
        n = V.norm(dim=-1, keepdim=True)
        V = V * (max_step / n.clamp_min(max_step))
    return V

# Part 2 — Target distribution and drift field

The target is a mixture of **4 Gaussian blobs**. We **superimpose** the base (green) and kernel-gradient (blue) unit drift directions on the same axes. Press ▶ below to **sweep the kernel temperature τ**: the two fields nearly coincide at small τ, but as τ grows the base direction is biased toward distant mass — exactly the bias the gradient fixes (paper Fig. 2).

In [ ]:
# ============================================================
# Define target distribution
# ============================================================
def make_gmm(n_modes=4, radius=0.7, std=0.1):
    ang = torch.arange(n_modes) * (2 * math.pi / n_modes)
    centers = torch.stack([radius * torch.cos(ang), radius * torch.sin(ang)], dim=1)

    def sampler(n, seed=None):
        g = torch.Generator().manual_seed(seed) if seed is not None else None
        idx = torch.randint(0, n_modes, (n,), generator=g)
        return centers[idx] + std * torch.randn(n, 2, generator=g)

    return sampler

GMM_SAMPLER = make_gmm()
TARGET = GMM_SAMPLER(5000, seed=0)
LIM = 1.2
g1 = torch.linspace(-LIM, LIM, 17)
gx, gy = torch.meshgrid(g1, g1, indexing="xy")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)

# ============================================================
# Visualize the drift field — sweep the kernel temperature τ
# ============================================================
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

def draw(temp):
    temp = float(temp)
    ax.clear()
    ax.scatter(TARGET[:, 0], TARGET[:, 1], s=4, alpha=0.25, c=COLOR_REF)
    for mode, color in [("base", COLOR_BASE), ("gradient", COLOR_GRAD)]:
        V = compute_V(grid, TARGET, grid, temp=temp, mode=mode)  # X_neg = grid
        U = V / V.norm(dim=-1, keepdim=True).clamp_min(1e-8)
        ax.quiver(grid[:, 0], grid[:, 1], U[:, 0], U[:, 1], angles="xy",
                  color=color, alpha=0.8, width=0.004,
                  label=("base" if mode == "base" else "gradient (ours)"))
    ax.set_aspect("equal"); ax.legend(loc="upper right", fontsize=8)
    ax.set_title(f"base (green) vs kernel-gradient (blue)   τ = {temp:.2f}")
    ax.set_xlim(-LIM, LIM); ax.set_ylim(-LIM, LIM)
    ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(figsize=(6.0, 6.0))
anim = FuncAnimation(fig, draw, frames=torch.linspace(0.05, 1.0, 20), interval=250)
plt.close(fig) 
HTML(anim.to_jshtml())

# Part 3 — Train and compare

In [ ]:
# ============================================================
# Backbone model
# ============================================================

class MLP(nn.Module):
    def __init__(self, in_dim=3, hidden=16, out_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, z):
        return self.net(z)

# ============================================================
# Comparison metric: Sliced Wasserstein-2 distance 
# ============================================================

@torch.no_grad()
def sliced_w2(x, y, n_proj=128, seed=0):
    '''Sliced Wasserstein-2 in R^D:
    project on random unit directions, sort, average the per-direction 1-D W2.'''
    x = torch.as_tensor(x).float(); y = torch.as_tensor(y).float()
    n = min(x.shape[0], y.shape[0]); x, y = x[:n], y[:n]
    gn = torch.Generator().manual_seed(seed)
    proj = torch.randn(n_proj, x.shape[1], generator=gn)
    proj = proj / proj.norm(dim=-1, keepdim=True).clamp_min(1e-12)
    xs, _ = (x @ proj.t()).sort(dim=0)
    ys, _ = (y @ proj.t()).sort(dim=0)
    return float(((xs - ys) ** 2).mean(dim=0).sqrt().mean().item())

# ============================================================
# Training loop
# ============================================================

def train_toy(mode, *, steps=3000, batch=2048, in_dim=3,
              hidden=16, lr=1e-3, seed=42, eval_every=100,
              snapshot_steps=(0, 50, 500, 1000, 2000, 3000)):
    torch.manual_seed(seed)
    model = MLP(in_dim, hidden, 2).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    drift = partial(compute_V, mode=mode,
                    temp=1.0, max_step=0.5)

    loss_hist, curve, snaps = [], [], {}
    if 0 in snapshot_steps:                       # pre-training (untrained net)
        with torch.no_grad():
            snaps[0] = model(torch.randn(5000, in_dim, device=DEVICE)).cpu().numpy()
    pbar = tqdm(range(1, steps + 1), desc=f"{mode:8s}")
    for step in pbar:
        pos = GMM_SAMPLER(batch).to(DEVICE)
        gen = model(torch.randn(batch, in_dim, device=DEVICE))
        loss = drifting_loss(gen, pos, drift)
        opt.zero_grad(); loss.backward(); opt.step()

        loss_hist.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.2e}")

        if step % eval_every == 0 or step in snapshot_steps:
            with torch.no_grad():
                s = model(torch.randn(5000, in_dim, device=DEVICE)).cpu()
            curve.append({"step": step, "sw": sliced_w2(s, TARGET)})
            if step in snapshot_steps:
                snaps[step] = s.numpy()
    return {"loss": loss_hist, "curve": curve, "snaps": snaps}

In [ ]:
# ============================================================
# Train the models
# ============================================================

res_base = train_toy("base")
res_grad = train_toy("gradient")

### Training evolution 

In [ ]:
# ============================================================
# Visualize evolution of the distributions during training
# ============================================================

steps = [0, 50, 500, 1000, 2000, 3000]
rows = [(res_base, "base", COLOR_BASE), (res_grad, "gradient", COLOR_GRAD)]
ncols = len(steps) + 1                                   # + target column on the right
fig, axes = plt.subplots(len(rows), ncols,
                         figsize=(1.7 * ncols, 1.7 * len(rows)))
for row, (res, name, c) in enumerate(rows):
    for col in range(ncols):
        a = axes[row, col]
        if col < len(steps):
            pts, color, head = res["snaps"][steps[col]], c, f"step {steps[col]}"
        else:
            pts, color, head = TARGET, COLOR_REF, "target"
        a.scatter(pts[:, 0], pts[:, 1], s=2, alpha=0.35, c=color, linewidths=0)
        a.set_aspect("equal"); a.set_xlim(-LIM, LIM); a.set_ylim(-LIM, LIM)
        a.axis("off")                                   
        if row == 0:
            a.text(0.5, 1.05, head, transform=a.transAxes,
                   ha="center", va="bottom", fontsize=10)
        if col == 0:
            a.text(-0.06, 0.5, name, transform=a.transAxes, rotation=90,
                   ha="center", va="center", fontsize=11, color=c)
fig.suptitle("Distribution evolution during training", y=1.02, fontsize=12)
plt.subplots_adjust(wspace=0.05, hspace=0.05)
plt.show()

### Metrics over training

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for res, name, col in [(res_base, "base", COLOR_BASE),
                       (res_grad, "gradient (ours)", COLOR_GRAD)]:
    xs = [r["step"] for r in res["curve"]]
    ax[0].plot(res["loss"], alpha=0.6, label=name, color=col)
    ax[1].plot(xs, [r["sw"] for r in res["curve"]], marker=".", label=name, color=col)
ax[0].set_yscale("log"); ax[0].set_title("training loss"); ax[0].set_xlabel("step")
ax[1].set_yscale("log"); ax[1].set_title("sliced Wasserstein-2"); ax[1].set_xlabel("step")
for a in ax:
    a.grid(True, alpha=0.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

Both formulations eventually recover all four modes, but the kernel-gradient drift gets there faster and cleaner, and its sliced-W2 stays lower throughout training. 